In [ ]:
"""
edit_coco_tracks.py
--------------------
Outil pour filtrer et renommer des tracks dans un fichier d'annotations COCO.
Les parametres sont lus depuis un fichier galaxy_inputs/galaxy_inputs.json.

Structure attendue du galaxy_inputs.json :
{
    "input_json":   "annotations.json",
    "mode":         "keep",                            // "keep" ou "remove"
    "id":    "0,1,2",                                  // optionnel (absent ou null)
                                                       // keep=tout garder, remove=ne rien supprimer
    "rename":       "1:Aplysia_juvenile,2:Aplysia_adulte"  // optionnel, peut etre "" ou absent
}

Note : les annotations sans champ "track_id" sont toujours conservees sans modification.
"""

import json
import sys
from pathlib import Path
from copy import deepcopy


def load_galaxy_inputs(path: str = "galaxy_inputs/galaxy_inputs.json") -> dict:
    p = Path(path)
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def parse_track_ids(raw: str) -> set | None:
    """Parse '0,1,3' -> {0, 1, 3}. Retourne None si la chaine est vide."""
    raw = raw.strip()
    if not raw:
        return None
    return {int(x.strip()) for x in raw.split(",") if x.strip() != ""}


def parse_renames(raw: str) -> dict:
    """Parse '0:Aplysia_adulte,1:Aplysia_juvenile' -> {0: 'Aplysia_adulte', 1: 'Aplysia_juvenile'}"""
    result = {}
    for item in raw.split(","):
        item = item.strip()
        if not item:
            continue
        if ":" not in item:
            raise ValueError(f"Format invalide pour rename : '{item}' (attendu track_id:nom)")
        track_id_str, new_name = item.split(":", 1)
        result[int(track_id_str.strip())] = new_name.strip()
    return result


def edit_coco(data: dict, mode: str, track_ids: set | None, renames: dict) -> dict:
    output = deepcopy(data)
    annotations = output.get("annotations", [])

    # Separer les annotations avec/sans track_id
    with_track    = [ann for ann in annotations if ann.get("track_id") is not None]
    without_track = [ann for ann in annotations if ann.get("track_id") is None]
    if without_track:
        print(f"  {len(without_track)} annotation(s) sans track_id conservee(s) sans modification.")

    # 1. Info sur les tracks presents
    all_track_ids = {ann["track_id"] for ann in with_track}
    print(f"Track IDs presents dans le fichier : {sorted(all_track_ids)}")

    # 2. Filtrage
    if mode == "keep":
        if track_ids is None:
            print("  Aucun track ID specifie en mode 'keep' - toutes les annotations sont conservees.")
        else:
            unknown = track_ids - all_track_ids
            if unknown:
                print(f"  Track IDs introuvables (keep) : {unknown}")
            with_track = [ann for ann in with_track if ann["track_id"] in track_ids]
            print(f"  Tracks gardes : {sorted(track_ids & all_track_ids)}")

    elif mode == "remove":
        if track_ids is None:
            print("  Aucun track ID specifie en mode 'remove' - aucune annotation supprimee.")
        else:
            unknown = track_ids - all_track_ids
            if unknown:
                print(f"  Track IDs introuvables (remove) : {unknown}")
            with_track = [ann for ann in with_track if ann["track_id"] not in track_ids]
            print(f"  Tracks supprimes : {sorted(track_ids & all_track_ids)}")

    else:
        print(f"Mode inconnu : '{mode}'. Utilisez 'keep' ou 'remove'.", file=sys.stderr)
        sys.exit(1)

    annotations = with_track + without_track

    # 3. Renommage des categories
    # Logique : si le nouveau nom existe deja -> reutiliser cette categorie
    #           sinon -> creer une nouvelle categorie
    # Dans les deux cas, on met a jour le category_id des annotations du track
    categories = output.get("categories", [])
    name_to_cat_id = {cat["name"]: cat["id"] for cat in categories}
    max_cat_id = max((cat["id"] for cat in categories), default=0)

    track_to_new_cat = {}
    for track_id, new_name in renames.items():
        if track_id not in all_track_ids:
            print(f"  Track ID {track_id} (rename -> '{new_name}') introuvable dans le fichier - ignore.")
            continue
        if new_name not in name_to_cat_id:
            max_cat_id += 1
            categories.append({"id": max_cat_id, "name": new_name, "supercategory": ""})
            name_to_cat_id[new_name] = max_cat_id
            print(f"  Nouvelle categorie creee : id={max_cat_id}, name='{new_name}'")
        else:
            print(f"  Categorie existante reutilisee : id={name_to_cat_id[new_name]}, name='{new_name}'")
        track_to_new_cat[track_id] = name_to_cat_id[new_name]

    for ann in annotations:
        track_id = ann.get("track_id")
        if track_id is not None and track_id in track_to_new_cat:
            old_cat = ann["category_id"]
            ann["category_id"] = track_to_new_cat[track_id]
            if old_cat != ann["category_id"]:
                print(f"   Track {track_id} : category_id {old_cat} -> {ann['category_id']}")

    # 4. Nettoyer les categories non utilisees
    used_cat_ids = {ann["category_id"] for ann in annotations}
    categories = [cat for cat in categories if cat["id"] in used_cat_ids]

    # 5. Renumeroter les annotations
    for new_id, ann in enumerate(annotations, start=1):
        ann["id"] = new_id

    output["annotations"] = annotations
    output["categories"] = categories
    print(f"\nResultat : {len(annotations)} annotation(s) conservee(s).")
    return output


def main():
    galaxy = load_galaxy_inputs("galaxy_inputs/galaxy_inputs.json")

    input_json    = galaxy["Annotation"][0]["path"]
    mode          = galaxy.get("mode", "").strip().lower()
    track_ids_raw = (galaxy.get("id") or "").strip()
    rename_raw    = (galaxy.get("rename") or "")

    track_ids = parse_track_ids(track_ids_raw)
    renames   = parse_renames(rename_raw) if rename_raw.strip() else {}

    print(f"  path    : {input_json}")
    print(f"  Mode    : {mode}")
    print(f"  Tracks  : {sorted(track_ids) if track_ids is not None else '(non specifie)'}")
    if renames:
        print(f"  Rename  : {renames}")

    input_path = Path(input_json)
    if not input_path.exists():
        print(f"Fichier COCO introuvable : {input_path}", file=sys.stderr)
        sys.exit(1)

    with open(input_path, encoding="utf-8") as f:
        data = json.load(f)

    result = edit_coco(data, mode, track_ids, renames)

    output_path = Path("outputs") / (input_path.stem + "_edited.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"Fichier sauvegarde : {output_path}")


if __name__ == "__main__":
    main()